# Packages import

In [36]:
import os
import json
import requests
from bs4 import BeautifulSoup

# Ceneo Scraper

1. provide URL address of product's opinions webpage


In [37]:
product_code="189795038"
page=1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send the request to provided URL address

In [38]:
response=requests.get(url)
print(response.status_code)

200


3. If status code is ok, fetch all opinions from requested webpage

In [39]:
page_dom= BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


4. For all fetched opinions, parse them to extract relevant data

In [40]:
product_name=page_dom.find("h1").get_text()
print(product_name)
print(type(product_name))

Onyx Boox Note Air 5 C
<class 'str'>


4. For all fetched opinions, parse them to extract relevant data

In [41]:
##opinions=page_dom.find_all('div', class_="js_product-review")
#print(type(opinions))
#print(len(opinions))


### opinions=page_dom.select('div.user-post_post')

In [42]:
opinions=page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))


### opinions=page_dom.select('div.user-post_post')

<class 'bs4.element.ResultSet'>
10


In [43]:
all_reviews = page_dom.find_all('div', class_="js_product-review")
# Remove the dot from the string check
opinions = [r for r in all_reviews if 'user-post--highlight' not in r.get('class', [])]
print(len(opinions))

10


In [44]:
# Find all potential reviews
#all_reviews = page_dom.find_all('div', class_="js_product-review")

# Filter: Keep it only if 'other-class' is NOT in the element's class list
#opinions = [r for r in all_reviews if 'user-post--highlight' not in r.get('class', [])]

#print(len(opinions))

In [45]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        "opinion_id":opinion["data-entry-id"],
        "author":opinion.select_one("span.user-post__author-name").get_text().strip(),
        'reccomendation':opinion.select_one('span.user-post__author-recommendation>em').get_text().strip() if opinion.
            select_one('span.user-post__author-recommendation>em') else None,
        "score":opinion.select_one('.user-post__score-count').get_text().strip(),
        "content":opinion.select_one('div.user-post__text').get_text().strip(),
        "pros":[p.get_text().strip() for p in opinion.select('div.review-feature__item--positive')],
        "cons":[c.get_text().strip() for c in opinion.select('div.review-feature__item--negative')],
        "like":opinion.select_one('button.vote-yes > span').get_text().strip(),
        "dislike":opinion.select_one('button.vote-no > span').get_text().strip(),
        "publishing_date":opinion.select_one('span.user-post__published > time:nth-child(1)')["datetime"].strip() if opinion.select_one('span.user-post__published > time:nth-child(1)') else None,
        "purchase_date":opinion.select_one('span.user-post__published > time:nth-child(2)')['datetime'].strip() if opinion.select_one('span.user-post__published > time:nth-child(2)') else None,
    }
    all_opinions.append(single_opinion)
    
json.dump(all_opinions,open("all_opinions.json",'w'),indent=4)

6.

In [46]:
next = True if page_dom.select_one("button.pagination__next") else False
if next: page+=1

In [47]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [48]:
with open(f"./opinions/{product_code}.json",'w',encoding="UTF-8") as file:
    json.dump(all_opinions,file,indent=4)